# Create DBU Awards (Deutsche Bundesstiftung Umwelt)

Creates Deutsche Bundesstiftung Umwelt (German Federal Environmental Foundation) awards
from the foundation's own public databases, read through its WordPress REST API
(`https://www.dbu.de/wp-json/wp/v2/...`):

- `projektdatenbank`: every funded project since 1991 (Aktenzeichen, title, grantee
  organisation, amount in EUR, project period, Bundesland, DBU theme tags; abstract for ~47%)
- `promotionsstipendium`: DBU PhD scholarships since 1992 (scholar, host institution,
  supervisor, period, abstract)
- `moe-fellowship`: MOE fellowships since 1998 (graduates from Central/Eastern Europe
  placed at German environmental research institutions, agencies and companies)

**14,184 records in the parquet; 13,490 shipped** (10,301 projects + 90 Green Start-up
grants + 1,753 PhD scholarships + 1,346 MOE fellowships; local run 2026-10-01).

**Scope filter (Kyle, 2026-10-01: keep every research pathway incl. scholarships,
fellowships and start-up grants; drop only what is clearly not research).** The
project database covers all DBU funding, including environmental education. A project
is EXCLUDED (`scope = 'excluded_environmental_education'`, 694 rows, EUR 108.5M) only if
(a) DBU's own theme tags are exactly `Umweltkommunikation` (no research, technology,
climate, resource, land-use, conservation or cultural-heritage theme), AND (b) the
grantee is not a university / Hochschule / research institute, AND (c) the title has
no research/development term (Forschung, Untersuchung, Studie, Analyse, Evaluation,
Monitoring, Erprobung, Bewertung, Konzept, Methode, Verfahren, Technologie, Entwicklung).
What this drops: school and kindergarten programmes, exhibitions, conferences,
environmental-centre construction, TV/radio/theatre productions. Everything else is kept,
including company demonstration projects (DBU funds only innovative model projects,
no plain equipment subsidies were found: 2 of 11,085 titles mention a purchase) and the
Green Start-up programme (AZ 35500-35509, EUR ~125k per start-up), flagged in the batch
report. The excluded rows stay in `dbu_raw` for audit.

**Amounts:** projects only (`bsumme`, German format, EUR); scholarships and
fellowships publish no amount (NULL).

**Investigators:** projects publish the grantee organisation but no person, so
`lead_investigator` carries the organisation only (given/family NULL, affiliation.name
= grantee; CORDIS / Foerderkatalog precedent). Scholarships and fellowships carry the
scholar as lead_investigator with the host institution as affiliation.

**Prerequisites:**
- Run `scripts/local/dbu_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/dbu/dbu_projects.parquet` (section 1.4 shrink guard).

**`funder_award_id` (section 2.1.1):** the DBU Aktenzeichen exactly as DBU prints it:
`34025/01` (projects, NNNNN/NN), `20013/247` (scholarships and fellowships,
NNNNN/NNN). Citation-side sample (`openalex_awards_raw`, F4320309954, priority 0):
the bare form dominates (`34025/01`, `20020/696`); `AZ 34025/01` and
`34123/01-33/2` (with DBU's internal Referat suffix) variants do not collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320309954`
- display_name: Deutsche Bundesstiftung Umwelt
- ror_id / doi: from `openalex.funders.funders` (ror 05pmq5w14, doi 10.13039/100007636)

**Priority:** `528` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dbu_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/dbu/dbu_projects.parquet`;

In [ ]:
%sql
SELECT record_type, programme, scope, COUNT(*) as n
FROM openalex.awards.dbu_raw
GROUP BY 1, 2, 3 ORDER BY 1, 2, 3;

In [ ]:
%sql
DESCRIBE openalex.awards.dbu_raw;

In [ ]:
%sql
SELECT funder_award_id, record_type, programme, title, org_name, amount, start_date, end_date,
       given_name, family_name, theme_tags, scope
FROM openalex.awards.dbu_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320309954 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309954;

## Step 2: Create DBU Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dbu_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309954  -- Deutsche Bundesstiftung Umwelt
),

src AS (
    SELECT *,
        named_struct(
            'given_name', CASE WHEN record_type <> 'project' THEN NULLIF(TRIM(given_name), '') END,
            'family_name', CASE WHEN record_type <> 'project' THEN NULLIF(TRIM(family_name), '') END,
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(regexp_replace(org_full, '\\s*\\n\\s*', ' ')), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead
    FROM openalex.awards.dbu_raw
    WHERE scope = 'in'  -- environmental-education-only projects excluded (see header)
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- 12 early MOE fellowship posts have the placeholder title '.' -> NULL
    CASE WHEN g.title RLIKE '[A-Za-z]' THEN TRIM(g.title) END as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    -- 4 projects carry bsumme 0,00 (no amount recorded) -> NULL
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE g.programme
        WHEN 'Promotionsstipendium' THEN 'studentship'
        WHEN 'MOE-Fellowship' THEN 'fellowship'
        WHEN 'Green Start-up' THEN 'innovation'
        ELSE 'research'
    END as funding_type,
    g.programme as funder_scheme,
    'dbu_projektdatenbank' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    CASE WHEN g.lead.family_name IS NOT NULL OR g.lead.affiliation.name IS NOT NULL THEN g.lead END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead.family_name IS NOT NULL OR g.lead.affiliation.name IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is a DBU Aktenzeichen; 0 rows expected.
SELECT funder_award_id FROM openalex.awards.dbu_awards
WHERE NOT funder_award_id RLIKE '^[0-9]{5}/[0-9]{2,3}(_[A-Za-z])?$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dbu_projektdatenbank' AND priority = 528;

-- Insert into openalex_awards_raw with priority.
-- Priority 528: direct-from-funder ingest of DBU's own project and scholarship databases.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 528 outranks the
-- acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    528 as priority
FROM openalex.awards.dbu_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_dbu_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.dbu_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funding_type, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.dbu_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.dbu_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.dbu_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check (scholars/fellows): top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.dbu_awards
WHERE lead_investigator.family_name IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.4a for projects: grantee organisations (repeat grantees are expected).
SELECT lead_investigator.affiliation.name AS grantee, COUNT(*) AS n
FROM openalex.awards.dbu_awards
WHERE funder_scheme IN ('Projektförderung', 'Green Start-up')
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.4a: titles should be distinct.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.dbu_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_person,
    COUNT(lead_investigator.affiliation.name) as has_org
FROM openalex.awards.dbu_awards;

In [ ]:
%sql
-- Acknowledgement shells (priority 0) that this ingest upgrades.
SELECT c.provenance, COUNT(*) as shell_rows, COUNT(t.id) as upgraded
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.dbu_awards t ON t.id = c.id
WHERE c.funder_id = 4320309954 AND c.priority = 0
GROUP BY c.provenance ORDER BY shell_rows DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dbu_projektdatenbank'
GROUP BY provenance, priority;